# Занятие 3. OCR для открытого скана на малоресурсном языке России

**Цель практики:** взять открытое изображение из Wikimedia Commons, прогнать лёгкий OCR baseline и понять, получается ли текст на вменяемом языке.

Пример использует материалы категории **Udmurt Dunne** на Wikimedia Commons и Tesseract с русской OCR-моделью как baseline для кириллического удмуртского текста. Это не “правильная удмуртская OCR-модель”, а проверка: насколько далеко можно зайти простым бесплатным инструментом.

Работает в бесплатном Colab на CPU.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-rus
!pip -q install pytesseract pillow opencv-python-headless pandas matplotlib

In [ ]:
import os, re, json, textwrap, math, statistics, random, io
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from PIL import Image
import pytesseract
import matplotlib.pyplot as plt
import cv2

## 1. Находим файл в Wikimedia Commons через API

In [ ]:
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
FILE_TITLE = 'File:Удномер.jpg'  # из категории Wikimedia Commons: Udmurt Dunne

params = {
    'action': 'query',
    'titles': FILE_TITLE,
    'prop': 'imageinfo',
    'iiprop': 'url|mime|size|extmetadata',
    'format': 'json',
}
r = requests.get(COMMONS_API, params=params, timeout=30)
r.raise_for_status()
page = next(iter(r.json()['query']['pages'].values()))
info = page['imageinfo'][0]
image_url = info['url']
print(image_url)
print('license:', info.get('extmetadata', {}).get('LicenseShortName', {}).get('value'))

## 2. Скачиваем изображение и смотрим на него

In [ ]:
img_path = DATA_DIR / 'udmurt_sample.jpg'
img_path.write_bytes(requests.get(image_url, timeout=60).content)
img = Image.open(img_path)
print(img.size)
plt.figure(figsize=(6, 8))
plt.imshow(img)
plt.axis('off');

## 3. Запускаем Tesseract baseline

In [ ]:
raw_text = pytesseract.image_to_string(img, lang='rus')
print(raw_text[:2000])
save_artifact('lesson03_ocr_raw.txt', raw_text)

## 4. Улучшаем препроцессинг и сравниваем

In [ ]:
gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
gray = cv2.fastNlMeansDenoising(gray, h=20)
thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
prep_path = DATA_DIR / 'udmurt_sample_preprocessed.png'
cv2.imwrite(str(prep_path), thr)

prep_img = Image.open(prep_path)
prep_text = pytesseract.image_to_string(prep_img, lang='rus')
print(prep_text[:2000])
save_artifact('lesson03_ocr_preprocessed.txt', prep_text)

plt.figure(figsize=(6, 8))
plt.imshow(prep_img, cmap='gray')
plt.axis('off');

## 5. Быстрая диагностика качества без ground truth

In [ ]:
def ocr_diagnostics(text):
    chars = len(text)
    letters = re.findall(r'[А-Яа-яЁёӐ-ӿ]', text)
    tokens = re.findall(r'[А-Яа-яЁёӐ-ӿ]{2,}', text)
    weird = re.findall(r'[^А-Яа-яЁёӐ-ӿ0-9\s.,:;!?()\-—"«»]', text)
    return {
        'chars': chars,
        'cyrillic_letters': len(letters),
        'tokens_2plus': len(tokens),
        'unique_tokens_2plus': len(set(t.lower() for t in tokens)),
        'weird_char_share': len(weird) / max(chars, 1),
    }

diag = pd.DataFrame([
    {'version': 'raw', **ocr_diagnostics(raw_text)},
    {'version': 'preprocessed', **ocr_diagnostics(prep_text)},
])
show_df(diag)
save_artifact('lesson03_ocr_diagnostics.csv', diag)

## Вопросы для отчёта

1. Можно ли читать результат глазами? Какие слова/буквы распознаются хуже всего?
2. Улучшил ли препроцессинг результат?
3. Почему русская OCR-модель может ошибаться на удмуртском?
4. Какие 30-50 строк стоит вручную разметить как ground truth для следующего шага?